In [1]:
import os

# Wipe out any potentially hijacked local configurations
vars_to_clear = ["OLLAMA_HOST", "http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY"]
for var in vars_to_clear:
    os.environ.pop(var, None)

# Force the environment to look at the official default port
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"


In [2]:
import sys
import re
import time
import zipfile
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from bs4 import BeautifulSoup
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from datetime import datetime, timezone
from langchain_chroma import Chroma
import hashlib
import xml.etree.ElementTree as ET
print(sys.executable)
print(sys.version)
print("requests", requests.__version__)

c:\Developer\MediBot\.venv\Scripts\python.exe
3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
requests 2.34.2


In [3]:

DATA_DIR = Path("medical_data")
DATA_DIR.mkdir(exist_ok=True)
XML_ZIP_PATH = DATA_DIR / "mplus_topics.zip"
XML_PATH = DATA_DIR / "mplus_topics.xml"

DB_DIR = "./medical_chroma_db"

# 2. Dynamically fetch the current download link for the compressed XML
HUB_URL = "https://medlineplus.gov/xml.html"
headers = {"User-Agent": "MedicalRAGResearch/0.1"}

print("Fetching latest download link...")
hub_response = requests.get(HUB_URL, timeout=30, headers=headers)
hub_response.raise_for_status()

soup = BeautifulSoup(hub_response.text, 'html.parser')

# Find the first link matching the exact anchor text visible on the page
download_link = None
for link in soup.find_all('a'):
    if link.text and "MedlinePlus Compressed Health Topic XML" in link.text:
        download_link = link.get('href')
        break  # Grab the most recent one generated at the top of the list

if not download_link:
    raise RuntimeError(
        "Could not find the 'MedlinePlus Compressed Health Topic XML' anchor text on the page. "
        "The layout may have changed."
    )

# Handle both relative paths (/xml/...) and absolute URLs safely
if download_link.startswith('http'):
    DOWNLOAD_URL = download_link
else:
    DOWNLOAD_URL = f"https://medlineplus.gov{download_link if download_link.startswith('/') else '/' + download_link}"

print(f"Successfully resolved target URL: {DOWNLOAD_URL}")

# 3. Download the actual zip archive
print("Downloading archive payload...")
response = requests.get(DOWNLOAD_URL, timeout=120, headers=headers)
response.raise_for_status()
XML_ZIP_PATH.write_bytes(response.content)

# 4. Extract the compressed XML contents out of the ZIP container
print("Extracting payload...")
with zipfile.ZipFile(XML_ZIP_PATH, "r") as archive:
    # Dynamically find the file ending with .xml inside the archive
    xml_filenames = [f for f in archive.namelist() if f.endswith(".xml")]
    if not xml_filenames:
        raise FileNotFoundError("No XML payload discovered inside the downloaded zip container.")
    
    extracted_data = archive.read(xml_filenames[0])
    XML_PATH.write_bytes(extracted_data)

print("\nProcessing complete!")
print(f"Extracted File: {XML_PATH}")
print(f"File size: {XML_PATH.stat().st_size} bytes")


EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

Fetching latest download link...


Successfully resolved target URL: https://medlineplus.gov/xml/mplus_topics_compressed_2026-10-27.zip
Extracting payload...

Processing complete!
Extracted File: medical_data\mplus_topics.xml
File size: 30154036 bytes


In [4]:
tree = ET.parse(XML_PATH)
root = tree.getroot()

medical_records = []

for topic in root.findall(".//health-topic"):

    title = topic.get("title", "")
    url = topic.get("url", "")
    language = topic.get("language", "")
    topic_id = topic.get("id", "")

    summary_element = topic.find("full-summary")

    if summary_element is not None:
        summary = " ".join(
            "".join(summary_element.itertext()).split()
        )
    else:
        summary = ""

    if not title or not summary or not url:
        continue

    if language != "English":
        continue

    medical_records.append({
        "topic_id": topic_id,
        "title": title,
        "content": summary,
        "url": url,
        "language": language,
        "source": "MedlinePlus",
        "category": "general"
    })

print("Total health topics:", len(medical_records))

Total health topics: 1017


In [5]:
df = pd.DataFrame(medical_records)

display(
    df[["topic_id", "title", "language","content", "url"]].head(2)
)

print("Total records:", len(df))

,topic_id,title,language,content,url
0,6308,A1C,English,"<p>A1C is a blood test for <a href=""https://me...",https://medlineplus.gov/a1c.html
1,3061,Abdominal Pain,English,<p>Your abdomen extends from below your chest ...,https://medlineplus.gov/abdominalpain.html


Total records: 1017


In [6]:
keyword = "Parkinson"

matches = df[
    df["title"].str.contains(
        keyword,
        case=False,
        na=False
    )
]

display(matches[["title", "content"]])

,title,content
712,Parkinson's Disease,<p>Parkinson's disease (PD) is a type of <a hr...


In [7]:
# CLEAN RECORDS

def cleaned_records(text):
    soup = BeautifulSoup(text,"html.parser")

    cleaned = soup.get_text(" ", strip=True)

    cleaned = re.sub(r'\s+'," ", cleaned)

    return cleaned.strip()

# KEEP UNIQUE RECORDS
unique_record = {}

for record in medical_records:
    record["content"] = cleaned_records(record["content"])

    unique_record[record["url"]] = record

medical_records = list(unique_record.values())

print(len(medical_records))

    

1017


In [8]:
# Converting to Langchain Documents
documents = []
for record in medical_records:
    doc = Document(
        page_content=record["content"],
        metadata = {
            "topic_id": record["topic_id"],
            "title": record["title"],
            "source": record["source"],
            "source_url": record["url"],
            "category": record["category"],
            "language": record["language"],
            "retrieved_at": datetime.now(
                timezone.utc
            ).isoformat()

        }
    )

    documents.append(doc)

print(f"Approved documents: {len(documents)}")

Approved documents: 1017


In [9]:
# SPLIT THE DOCUMENTS INTO CHUNKS
text_splitters = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap=200,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " "
        ]
)

chunked_docs = text_splitters.split_documents(documents)
print(f"Total Chunks: {len(chunked_docs)}")

Total Chunks: 2902


In [10]:
# Initialization of Embeddings
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",  # or your preferred embedding model
    base_url="http://127.0.0.1:11434"  # Explicitly force the standard Ollama port
)



    

In [11]:
# Vector Store
vectorstore = Chroma(
    collection_name = "medical_data",
    embedding_function=embeddings,
    persist_directory=DB_DIR
)

In [14]:
# Inserting approved chunks
if chunked_docs:
    ids = [
        hashlib.sha256(
            (
                chunk.metadata["source_url"]
                + "|"
                + chunk.page_content
            ).encode("utf-8")
        ).hexdigest()
        for chunk in chunked_docs
    ]

    # 2. Define a small batch size to prevent overloading Ollama
    BATCH_SIZE = 32  
    
    print(f"Starting ingestion of {len(chunked_docs)} chunks...")
    
    # 3. Add documents in chunks sequentially
    for i in range(0, len(chunked_docs), BATCH_SIZE):
        batch_docs = chunked_docs[i : i + BATCH_SIZE]
        batch_ids = ids[i : i + BATCH_SIZE]


        vectorstore.add_documents(
            documents=batch_docs,
            ids=batch_ids
        )

    print("Knowledge base created!")
else:
    print("No approved chunks available to store.")

Starting ingestion of 2902 chunks...
Knowledge base created!


In [15]:
# Creating retriever
general_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {
            "category": "general"
        }
    }
)

In [16]:
# Testing Retrievel
question = "What are the common symptoms of diabetes?"

retrieved_docs = general_retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nDocument {i}")
    print("Title:", doc.metadata.get("title"))
    print("Source:", doc.metadata.get("source"))
    print("URL:", doc.metadata.get("source_url"))
    print("Content:", doc.page_content[:500])
    print("-" * 60)


Document 1
Title: Diabetes
Source: MedlinePlus
URL: https://medlineplus.gov/diabetes.html
Content: . Have polycystic ovary syndrome (PCOS). Are African American, Hispanic/Latino, American Indian, Alaska Native, Native Hawaiian, or Pacific Islander. What are the symptoms of diabetes? The symptoms of diabetes may include: Feeling very thirsty Feeling very hungry Urinating (peeing) more often, including at night Fatigue Blurry vision Numbness or tingling in the feet or hands Sores that do not heal Losing weight without trying But it's important to know that your symptoms may vary, depending on w
------------------------------------------------------------

Document 2
Title: Diabetes Type 1
Source: MedlinePlus
URL: https://medlineplus.gov/diabetestype1.html
Content: . What are the symptoms of type 1 diabetes? Symptoms often start quickly, especially in children. In adults, they may develop more slowly. Symptoms may include: Feeling very thirsty Feeling very hungry Urinating (peeing) more 